# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

df = pd.read_csv('../../data/starter.csv')

# Features we will use
X = df[['text_length', 'hashtags_count', 'author_follower_count', 'hour_of_day', 'content_type']].copy()

# Handle missing
imputer = SimpleImputer(strategy='median')
X[['author_follower_count']] = imputer.fit_transform(X[['author_follower_count']])

# Categorical
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat = enc.fit_transform(X[['content_type']])
X = X.drop('content_type', axis=1)
X = pd.concat([X, pd.DataFrame(cat, columns=enc.get_feature_names_out())], axis=1)

print("Feature vector shape:", X.shape)
X.head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

text_length: Number of characters in post text. Missing=0. Available at post time.
hashtags_count: Number of hashtags used. Missing=0. Available at post time.
author_follower_count: Author's followers. Missing=median fill. Available at post time.
hour_of_day: When post was published 0-23. No missing. Available at post time.
content_type: video/image/text. One-hot encoded. No missing. Available at post time.
All features are known BEFORE we predict engagement.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# Check for label leakage: any feature correlated >0.95 with label?
corr = df[['engagement_score_24h', 'text_length', 'hashtags_count', 'author_follower_count']].corr()
print(corr['engagement_score_24h'])

# Check future data: nothing uses likes_24h as feature
print("Leakage check: No future features used. All features are pre-publish.")

No leakage found. We avoided using final_like_count, final_share_count etc.
Only used data available at publish time.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded: user_email - PII, privacy risk, not predictive
Excluded: post_id - unique ID, causes overfitting, no generalization
Excluded: final_like_count - label leakage, only known after 24h
Excluded: ip_address - PII, not relevant for ranking

## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.